In [1]:
# ============================================================
# TIMEGPT CON VARIABLES EXÓGENAS
# BASE: unificada (2).csv
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nixtla import NixtlaClient
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

ARCHIVO = "unificada.csv"

FECHA = "fecha"

TARGET = "radiacion"


# ============================================================
# VARIABLES EXÓGENAS
# NOMBRES VERIFICADOS EN EL CSV
# ============================================================

EXOGENAS = [
    "temperatura",
    "humedad",
    "presion",
    "precipitacion",
    "veloc_de_viento_prom_10m"
]


# ============================================================
# PARÁMETROS MODIFICABLES
# ============================================================

FECHA_INICIO_TEST = "2026-02-15 00:00:00"

HORAS_PREDICCION = 168

DIAS_CONTEXTO = 630

FREQ = "15min"

PASOS_POR_HORA = 4

H = HORAS_PREDICCION * PASOS_POR_HORA


print("=" * 70)
print("TIMEGPT - RADIACIÓN CON VARIABLES EXÓGENAS")
print("=" * 70)

print("Fecha test       :", FECHA_INICIO_TEST)
print("Horizonte        :", HORAS_PREDICCION, "horas")
print("Pasos            :", H)
print("Contexto         :", DIAS_CONTEXTO, "días")
print("Objetivo         :", TARGET)

print("\nVariables exógenas:")

for variable in EXOGENAS:
    print(" -", variable)




TIMEGPT - RADIACIÓN CON VARIABLES EXÓGENAS
Fecha test       : 2026-02-15 00:00:00
Horizonte        : 168 horas
Pasos            : 672
Contexto         : 630 días
Objetivo         : radiacion

Variables exógenas:
 - temperatura
 - humedad
 - presion
 - precipitacion
 - veloc_de_viento_prom_10m


In [2]:
# ============================================================
# 2. API TIMEGPT
# ============================================================

#API_KEY = os.getenv("NIXTLA_API_KEY")
API_KEY = 'nixak-0dea77cc906e018c018d8bd6071c8c6cfb754654bb3adb5585155dbbcd7ba352cbcbefd609d95044'

if not API_KEY:
    raise ValueError(
        "No se encontró la variable NIXTLA_API_KEY"
    )

nixtla_client = NixtlaClient(
    api_key=API_KEY
)


# ============================================================
# 3. CARGAR CSV
# ============================================================

df = pd.read_csv(
    ARCHIVO,
    sep=";",
    low_memory=False
)


print("\nDimensión:", df.shape)

print("\nColumnas encontradas:")

for c in df.columns:
    print(c)



Dimensión: (336767, 19)

Columnas encontradas:
fecha
temperatura
humedad
presion
radiacion
precipitacion
precipitacion_acum_diaria
veloc_de_viento_max_10m
veloc_de_viento_min_10m
veloc_de_viento_prom_10m
direccion_de_viento_a_10m
direccion_a_10m
veloc_de_viento_max_2m
veloc_de_viento_min_2m
veloc_de_viento_prom_2m
direccion_de_viento_a_2m
direccion_a_2m
evaporimetro
evaporacion_acum_diaria


In [3]:
# ============================================================
# 4. VERIFICAR COLUMNAS
# ============================================================

columnas_necesarias = (
    [FECHA, TARGET]
    +
    EXOGENAS
)


faltantes = [
    c for c in columnas_necesarias
    if c not in df.columns
]


if faltantes:

    raise ValueError(
        f"Faltan columnas: {faltantes}"
    )


print("\nTodas las variables fueron encontradas.")


# ============================================================
# 5. FECHA
# ============================================================

df[FECHA] = pd.to_datetime(
    df[FECHA],
    dayfirst=True,
    errors="coerce"
)


# ============================================================
# 6. CONVERTIR A NUMÉRICO
# ============================================================

VARIABLES = [
    TARGET
] + EXOGENAS


for col in VARIABLES:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ============================================================
# 7. CONSERVAR VARIABLES
# ============================================================

df = df[
    [FECHA]
    +
    VARIABLES
].copy()


df = df.dropna(
    subset=[FECHA]
)


# ============================================================
# 8. DUPLICADOS
# ============================================================

print(
    "\nTimestamps duplicados:",
    df[FECHA].duplicated().sum()
)


df = (
    df.groupby(
        FECHA,
        as_index=False
    )[VARIABLES]
    .mean()
)


df = df.sort_values(FECHA)




Todas las variables fueron encontradas.

Timestamps duplicados: 3


In [4]:

# ============================================================
# 9. REGULARIZAR A 15 MINUTOS
# ============================================================

df = df.set_index(FECHA)


indice_completo = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq=FREQ
)


df = df.reindex(
    indice_completo
)


df.index.name = FECHA


print("\nFaltantes antes de interpolar:")

print(
    df[VARIABLES]
    .isna()
    .sum()
)


# ============================================================
# 10. INTERPOLAR HUECOS CORTOS
# ============================================================

for col in VARIABLES:

    df[col] = (
        df[col]
        .interpolate(
            method="time",
            limit=4
        )
    )


print("\nFaltantes después de interpolación corta:")

print(
    df[VARIABLES]
    .isna()
    .sum()
)


# ============================================================
# 11. COMPLETAR RESTANTES
# ============================================================

for col in VARIABLES:

    df[col] = (
        df[col]
        .interpolate(
            method="time",
            limit_direction="both"
        )
    )


print("\nFaltantes finales:")

print(
    df[VARIABLES]
    .isna()
    .sum()
)


df = df.reset_index()



Faltantes antes de interpolar:
radiacion                   13642
temperatura                 13467
humedad                     13443
presion                     15103
precipitacion               13381
veloc_de_viento_prom_10m    13342
dtype: int64

Faltantes después de interpolación corta:
radiacion                   12098
temperatura                 12090
humedad                     12089
presion                     13581
precipitacion               12085
veloc_de_viento_prom_10m    12082
dtype: int64

Faltantes finales:
radiacion                   0
temperatura                 0
humedad                     0
presion                     0
precipitacion               0
veloc_de_viento_prom_10m    0
dtype: int64


In [5]:
# ============================================================
# 12. VARIABLES TEMPORALES
# ============================================================

df["hora"] = (
    df[FECHA].dt.hour
    +
    df[FECHA].dt.minute / 60
)


df["dia_anio"] = (
    df[FECHA].dt.dayofyear
)


# Ciclo diario

df["hora_sin"] = np.sin(
    2 * np.pi *
    df["hora"] / 24
)

df["hora_cos"] = np.cos(
    2 * np.pi *
    df["hora"] / 24
)


# Ciclo anual

df["anio_sin"] = np.sin(
    2 * np.pi *
    df["dia_anio"] / 365.25
)

df["anio_cos"] = np.cos(
    2 * np.pi *
    df["dia_anio"] / 365.25
)


TEMPORALES = [
    "hora_sin",
    "hora_cos",
    "anio_sin",
    "anio_cos"
]


# ============================================================
# 13. DEFINIR PERÍODO
# ============================================================

fecha_inicio = pd.Timestamp(
    FECHA_INICIO_TEST
)


fecha_fin = (
    fecha_inicio
    +
    pd.Timedelta(
        minutes=15 * (H - 1)
    )
)


fecha_inicio_contexto = (
    fecha_inicio
    -
    pd.Timedelta(
        days=DIAS_CONTEXTO
    )
)


print("\nPERÍODO")

print(
    fecha_inicio,
    "->",
    fecha_fin
)





PERÍODO
2026-02-15 00:00:00 -> 2026-02-21 23:45:00


In [ ]:

# ============================================================
# 14. TRAIN
# ============================================================

train = df[
    (df[FECHA] >= fecha_inicio_contexto)
    &
    (df[FECHA] < fecha_inicio)
].copy()


# ============================================================
# 15. TEST
# ============================================================

test = df[
    (df[FECHA] >= fecha_inicio)
    &
    (df[FECHA] <= fecha_fin)
].copy()


print("\nTrain:", len(train))
print("Test :", len(test))


if len(test) != H:

    raise ValueError(
        f"El test debería tener {H} registros "
        f"pero tiene {len(test)}"
    )


# ============================================================
# 16. FORMATO TIMEGPT TRAIN
# ============================================================

train_tg = train.rename(
    columns={
        FECHA: "ds",
        TARGET: "y"
    }
)


train_tg["unique_id"] = "radiacion"


train_tg = train_tg[
    ["unique_id", "ds", "y"]
    +
    EXOGENAS
    +
    TEMPORALES
]


# ============================================================
# 17. EXÓGENAS FUTURAS
# ============================================================

X_df = test.rename(
    columns={
        FECHA: "ds"
    }
)


X_df["unique_id"] = "radiacion"


X_df = X_df[
    ["unique_id", "ds"]
    +
    EXOGENAS
    +
    TEMPORALES
]


print("\nVariables enviadas a TimeGPT:")

print(
    train_tg.columns.tolist()
)


print("\nExógenas:")

for c in EXOGENAS:
    print(" -", c)


# ============================================================
# 18. COMPROBAR DATOS
# ============================================================

print("\nNaN TRAIN:")

print(
    train_tg.isna().sum()
)


print("\nNaN X_df:")

print(
    X_df.isna().sum()
)


print(
    "\nFrecuencia:",
    pd.infer_freq(
        train_tg["ds"]
    )
)


# ============================================================
# 19. TIMEGPT
# ============================================================

print("\n" + "=" * 70)
print("EJECUTANDO TIMEGPT")
print("=" * 70)


forecast = nixtla_client.forecast(

    df=train_tg,

    X_df=X_df,

    h=H,

    freq=FREQ,

    time_col="ds",

    target_col="y",

    id_col="unique_id",

    level=[80]

)


print("\nPronóstico terminado.")

print(
    forecast.head()
)


# ============================================================
# 20. PREPARAR RESULTADOS
# ============================================================

forecast["ds"] = pd.to_datetime(
    forecast["ds"]
)


real = test[
    [FECHA, TARGET]
].copy()


real = real.rename(
    columns={
        FECHA: "ds",
        TARGET: "real"
    }
)


resultado = real.merge(
    forecast,
    on="ds",
    how="left"
)


resultado = resultado.rename(
    columns={
        "TimeGPT": "prediccion"
    }
)


# ============================================================
# 21. ERRORES
# ============================================================

resultado["error"] = (
    resultado["real"]
    -
    resultado["prediccion"]
)


resultado["error_absoluto"] = (
    np.abs(
        resultado["error"]
    )
)


# ============================================================
# 22. MÉTRICAS
# ============================================================

y_true = resultado["real"].to_numpy()

y_pred = resultado["prediccion"].to_numpy()


MAE = mean_absolute_error(
    y_true,
    y_pred
)


RMSE = np.sqrt(
    mean_squared_error(
        y_true,
        y_pred
    )
)


# WAPE

WAPE = (
    100
    *
    np.sum(
        np.abs(
            y_true - y_pred
        )
    )
    /
    np.sum(
        np.abs(y_true)
    )
)


# sMAPE

den = (
    np.abs(y_true)
    +
    np.abs(y_pred)
)


mask = den > 0


SMAPE = (
    100
    *
    np.mean(
        2
        *
        np.abs(
            y_true[mask]
            -
            y_pred[mask]
        )
        /
        den[mask]
    )
)


# ============================================================
# 23. MÉTRICAS DIURNAS
# ============================================================

mask_dia = (
    y_true > 20
)


MAE_DIA = mean_absolute_error(
    y_true[mask_dia],
    y_pred[mask_dia]
)


RMSE_DIA = np.sqrt(
    mean_squared_error(
        y_true[mask_dia],
        y_pred[mask_dia]
    )
)


# ============================================================
# 24. RESULTADOS
# ============================================================

print("\n" + "=" * 70)

print("RESULTADOS TIMEGPT + EXÓGENAS")

print("=" * 70)


print(
    f"MAE        : {MAE:.4f}"
)

print(
    f"RMSE       : {RMSE:.4f}"
)

print(
    f"sMAPE      : {SMAPE:.2f}%"
)

print(
    f"WAPE       : {WAPE:.2f}%"
)

print(
    f"MAE diurno : {MAE_DIA:.4f}"
)

print(
    f"RMSE diurno: {RMSE_DIA:.4f}"
)


# ============================================================
# 25. GRÁFICO
# ============================================================

plt.figure(
    figsize=(16, 6)
)


plt.plot(
    resultado["ds"],
    resultado["real"],
    label="Radiación real",
    linewidth=2
)


plt.plot(
    resultado["ds"],
    resultado["prediccion"],
    "--",
    label="TimeGPT + exógenas",
    linewidth=2
)


plt.title(
    "Radiación real vs TimeGPT con variables meteorológicas\n"
    f"Horizonte: {HORAS_PREDICCION} horas"
)


plt.xlabel(
    "Fecha y hora"
)

plt.ylabel(
    "Radiación"
)


plt.legend()

plt.grid(
    alpha=0.3
)

plt.xticks(
    rotation=45
)

plt.tight_layout()

plt.show()


# ============================================================
# 26. GRÁFICO ERROR
# ============================================================

plt.figure(
    figsize=(16, 5)
)


plt.plot(
    resultado["ds"],
    resultado["error"]
)


plt.axhline(
    0,
    linestyle="--"
)


plt.title(
    "Error TimeGPT con variables meteorológicas\n"
    f"Horizonte: {HORAS_PREDICCION} horas"
)


plt.xlabel(
    "Fecha y hora"
)

plt.ylabel(
    "Error (Real - Predicción)"
)


plt.grid(
    alpha=0.3
)

plt.xticks(
    rotation=45
)

plt.tight_layout()

plt.show()

INFO:nixtla.nixtla_client:Validating inputs...
INFO:nixtla.nixtla_client:Preprocessing dataframes...
INFO:nixtla.nixtla_client:Querying model metadata...



Train: 60480
Test : 672

Variables enviadas a TimeGPT:
['unique_id', 'ds', 'y', 'temperatura', 'humedad', 'presion', 'precipitacion', 'veloc_de_viento_prom_10m', 'hora_sin', 'hora_cos', 'anio_sin', 'anio_cos']

Exógenas:
 - temperatura
 - humedad
 - presion
 - precipitacion
 - veloc_de_viento_prom_10m

NaN TRAIN:
unique_id                   0
ds                          0
y                           0
temperatura                 0
humedad                     0
presion                     0
precipitacion               0
veloc_de_viento_prom_10m    0
hora_sin                    0
hora_cos                    0
anio_sin                    0
anio_cos                    0
dtype: int64

NaN X_df:
unique_id                   0
ds                          0
temperatura                 0
humedad                     0
presion                     0
precipitacion               0
veloc_de_viento_prom_10m    0
hora_sin                    0
hora_cos                    0
anio_sin                    0
